# ??????? ?? ?? ???

? ???? Isolation Forest, Risk LightGBM, Leadtime LightGBM? ?? ??????? ??? ??? ??? ????.

?? ??? ??? ??.

- train split??? ??
- validation split?? ??????? ??
- holdout split? ?? 1? ??
- ?? ?? ??? ???? ???? ??


In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px

REPORT_DIR = Path("experiment_comparison")
if not REPORT_DIR.exists():
    REPORT_DIR = Path("report/experiment_comparison")

def read_csv(name):
    path = REPORT_DIR / name
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

iforest = read_csv("hyperparameter_tuning_iforest_metrics.csv")
risk = read_csv("hyperparameter_tuning_risk_metrics.csv")
leadtime = read_csv("hyperparameter_tuning_leadtime_metrics.csv")
summary = json.loads((REPORT_DIR / "hyperparameter_tuning_summary.json").read_text(encoding="utf-8-sig"))
summary.keys()

dict_keys(['isolation_forest', 'risk_lgbm', 'leadtime_lgbm'])

## 1. Isolation Forest ?? ??

Isolation Forest? ?? ???? ??? window? ?? ??? ???? ????. ?? ????? ?? ?? ????.

- `n_estimators`
- `max_samples`
- `max_features`
- `bootstrap`
- `threshold_quantile`

??? ?? ? ??? `anomaly_score` ??? ???? `anomaly_label` ???? ?? ??? ???. ?? downstream? risk/priority? ?? ??? `anomaly_score`? ???, threshold ??? priority ????? anomaly label? ??? ?? ?? ???? ? ????? ????.


In [2]:
if not iforest.empty:
    iforest_holdout = iforest[iforest["split"].eq("holdout")].copy()
    display_iforest = iforest_holdout.sort_values("f1", ascending=False).head(10).rename(columns={
        "n_estimators": "?? ?",
        "max_samples": "?? ??",
        "max_features": "?? ??",
        "bootstrap": "?????",
        "threshold_quantile": "Threshold quantile",
        "precision": "???",
        "recall": "???",
        "f1": "F1",
        "false_positive_rate": "???",
        "true_positive_count": "TP",
        "false_positive_count": "FP",
    })
    display(display_iforest[["?? ?", "?? ??", "?? ??", "?????", "Threshold quantile", "???", "???", "F1", "???", "TP", "FP"]])

    fig = px.scatter(
        iforest_holdout,
        x="recall",
        y="false_positive_rate",
        color="threshold_quantile",
        size="f1",
        hover_data=["n_estimators", "max_samples", "max_features", "bootstrap", "precision", "f1"],
        title="Isolation Forest: Holdout ???? ??? trade-off",
        labels={"recall": "???", "false_positive_rate": "???", "threshold_quantile": "Threshold quantile", "f1": "F1"},
    )
    fig.show()


,?? ?,?? ??,?? ??,?? ??,?? ??,?????,Threshold quantile,???,???,???,???,???,???,F1,???,???,???,TP,FP
33,300,0.75,0.75,0.75,0.75,False,0.9,0.867925,0.345865,0.026820,0.867925,0.345865,0.026820,0.494624,0.867925,0.345865,0.026820,46,7
129,600,0.75,0.75,0.75,0.75,False,0.9,0.914894,0.323308,0.015326,0.914894,0.323308,0.015326,0.477778,0.914894,0.323308,0.015326,43,4
65,300,1.0,0.75,1.0,0.75,False,0.9,0.857143,0.315789,0.026820,0.857143,0.315789,0.026820,0.461538,0.857143,0.315789,0.026820,42,7
177,600,1.0,1.00,1.0,1.00,False,0.9,0.911111,0.308271,0.015326,0.911111,0.308271,0.015326,0.460674,0.911111,0.308271,0.015326,41,4
161,600,1.0,0.75,1.0,0.75,False,0.9,0.891304,0.308271,0.019157,0.891304,0.308271,0.019157,0.458101,0.891304,0.308271,0.019157,41,5
73,300,1.0,0.75,1.0,0.75,True,0.9,0.888889,0.300752,0.019157,0.888889,0.300752,0.019157,0.449438,0.888889,0.300752,0.019157,40,5
89,300,1.0,1.00,1.0,1.00,True,0.9,0.928571,0.293233,0.011494,0.928571,0.293233,0.011494,0.445714,0.928571,0.293233,0.011494,39,3
137,600,0.75,0.75,0.75,0.75,True,0.9,0.926829,0.285714,0.011494,0.926829,0.285714,0.011494,0.436782,0.926829,0.285714,0.011494,38,3
169,600,1.0,0.75,1.0,0.75,True,0.9,0.883721,0.285714,0.019157,0.883721,0.285714,0.019157,0.431818,0.883721,0.285714,0.019157,38,5
49,300,0.75,1.00,0.75,1.00,False,0.9,0.948718,0.278195,0.007663,0.948718,0.278195,0.007663,0.430233,0.948718,0.278195,0.007663,37,2


### Isolation Forest ??

?? ??? ?? ?????.

```text
?? holdout: F1 0.2267 / Recall 0.1278 / FPR 0.0000
?? holdout: F1 0.4615 / Recall 0.3158 / FPR 0.0268
```

?? ??? pre_fault ???? ?? ???? ??? ???. ??? ?? ????? `anomaly_label`? ? ???? ?? ?? ??? ??? ?? ??. ???? ?? 0? ??? ???? ?? threshold quantile ??? `q=0.92`? ? ???? ???.


## 2. Risk LightGBM ?? ??

Risk LightGBM? ????? ???? ? ?? ??? ???? ???? supervised ????.

?? ??? LightGBM? ?? ???? ??? validation?? ???? holdout?? ?? ????.


In [3]:
if not risk.empty:
    risk_val = risk[risk["split"].eq("validation")].copy()
    risk_hold = risk[risk["split"].eq("holdout")].copy()
    top_val = risk_val.sort_values("f1", ascending=False).head(10)[["candidate_id", "f1", "recall", "precision", "false_positive_rate", "roc_auc", "average_precision"]]
    top_hold = risk_hold.sort_values("f1", ascending=False).head(10)[["candidate_id", "f1", "recall", "precision", "false_positive_rate", "roc_auc", "average_precision"]]
    display(top_val.rename(columns={"candidate_id": "?? ID", "f1": "Validation F1", "recall": "???", "precision": "???", "false_positive_rate": "???", "roc_auc": "ROC-AUC", "average_precision": "AP"}))
    display(top_hold.rename(columns={"candidate_id": "?? ID", "f1": "Holdout F1", "recall": "???", "precision": "???", "false_positive_rate": "???", "roc_auc": "ROC-AUC", "average_precision": "AP"}))

    merged = risk_val[["candidate_id", "f1", "recall", "false_positive_rate"]].merge(
        risk_hold[["candidate_id", "f1", "recall", "false_positive_rate"]],
        on="candidate_id",
        suffixes=("_validation", "_holdout"),
    )
    fig = px.scatter(
        merged,
        x="f1_validation",
        y="f1_holdout",
        hover_data=["candidate_id", "recall_validation", "recall_holdout", "false_positive_rate_validation", "false_positive_rate_holdout"],
        title="Risk LightGBM: Validation F1 ?? Holdout F1",
        labels={"f1_validation": "Validation F1", "f1_holdout": "Holdout F1"},
    )
    fig.show()


,?? ID,Validation F1,???,???,???,ROC-AUC,AP
706,354,0.762963,0.72028,0.811024,0.083333,0.791934,0.803677
710,356,0.762963,0.72028,0.811024,0.083333,0.791934,0.803677
850,426,0.757353,0.72028,0.798450,0.090278,0.790380,0.804159
854,428,0.757353,0.72028,0.798450,0.090278,0.790380,0.804159
566,284,0.749091,0.72028,0.780303,0.100694,0.795455,0.808652
562,282,0.749091,0.72028,0.780303,0.100694,0.795455,0.808652
708,355,0.749091,0.72028,0.780303,0.100694,0.791715,0.791496
704,353,0.749091,0.72028,0.780303,0.100694,0.791715,0.791496
564,283,0.746377,0.72028,0.774436,0.104167,0.781153,0.795771
560,281,0.746377,0.72028,0.774436,0.104167,0.781153,0.795771


,?? ID,Holdout F1,???,???,???,ROC-AUC,AP
315,158,0.515337,0.488372,0.545455,0.163551,0.707890,0.471698
319,160,0.515337,0.488372,0.545455,0.163551,0.707890,0.471698
45,23,0.511364,0.523256,0.500000,0.210280,0.578624,0.332898
41,21,0.511364,0.523256,0.500000,0.210280,0.578624,0.332898
77,39,0.505882,0.500000,0.511905,0.191589,0.638339,0.391845
73,37,0.505882,0.500000,0.511905,0.191589,0.638339,0.391845
121,61,0.505882,0.500000,0.511905,0.191589,0.638339,0.391845
125,63,0.505882,0.500000,0.511905,0.191589,0.638339,0.391845
25,13,0.497110,0.500000,0.494253,0.205607,0.662247,0.401373
29,15,0.497110,0.500000,0.494253,0.205607,0.662247,0.401373


### Risk LightGBM ??

Risk ??? ?? ??? ? ??.

```text
?? holdout: F1 0.5466 / Recall 0.5116 / FPR 0.1449 / ROC-AUC 0.7628
?? holdout: F1 0.3250 / Recall 0.3023 / FPR 0.2243 / ROC-AUC 0.5592
```

validation?? ??? ??? validation F1? ???? holdout?? ?? ????. ?? ??????? ??? ??? holdout ??/?? ?? ??? ??? ??? ???. ??? Risk? ?? ?? calibrated ??? ???? ?? ??.


## 3. Leadtime LightGBM ?? ??

Leadtime ??? pre_fault row ??? ?? ? pseudo lead time bucket? ????. ?? ??? ?? 3?? ??? ??? ???? LightGBM ????? ????.


In [4]:
if not leadtime.empty:
    lead_val = leadtime[leadtime["split"].eq("validation")].copy()
    lead_hold = leadtime[leadtime["split"].eq("holdout")].copy()
    display(lead_val.sort_values("macro_f1", ascending=False).head(10)[["candidate_id", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]].rename(columns={
        "candidate_id": "?? ID", "accuracy": "???", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1", "top2_accuracy": "Top2 ???", "bucket_distance_mae": "???? MAE"
    }))
    display(lead_hold.sort_values("macro_f1", ascending=False).head(10)[["candidate_id", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]].rename(columns={
        "candidate_id": "?? ID", "accuracy": "???", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1", "top2_accuracy": "Top2 ???", "bucket_distance_mae": "???? MAE"
    }))

    merged = lead_val[["candidate_id", "macro_f1", "accuracy"]].merge(
        lead_hold[["candidate_id", "macro_f1", "accuracy"]],
        on="candidate_id",
        suffixes=("_validation", "_holdout"),
    )
    fig = px.scatter(
        merged,
        x="macro_f1_validation",
        y="macro_f1_holdout",
        hover_data=["candidate_id", "accuracy_validation", "accuracy_holdout"],
        title="Leadtime LightGBM: Validation Macro F1 ?? Holdout Macro F1",
        labels={"macro_f1_validation": "Validation Macro F1", "macro_f1_holdout": "Holdout Macro F1"},
    )
    fig.show()


,?? ID,???,Macro F1,Weighted F1,Top2 ???,???? MAE
858,430,0.503497,0.371477,0.455367,0.79021,0.622378
862,432,0.503497,0.371477,0.455367,0.79021,0.622378
786,394,0.503497,0.371477,0.455367,0.79021,0.622378
790,396,0.503497,0.371477,0.455367,0.79021,0.622378
620,311,0.496503,0.369312,0.449018,0.79021,0.643357
616,309,0.496503,0.369312,0.449018,0.79021,0.643357
548,275,0.496503,0.369312,0.449018,0.79021,0.643357
544,273,0.496503,0.369312,0.449018,0.79021,0.643357
1222,612,0.496503,0.367581,0.447244,0.79021,0.622378
1218,610,0.496503,0.367581,0.447244,0.79021,0.622378


,?? ID,???,Macro F1,Weighted F1,Top2 ???,???? MAE
621,311,0.72093,0.489306,0.710325,0.965116,0.313953
617,309,0.72093,0.489306,0.710325,0.965116,0.313953
549,275,0.72093,0.489306,0.710325,0.965116,0.313953
545,273,0.72093,0.489306,0.710325,0.965116,0.313953
163,82,0.72093,0.489076,0.710944,0.965116,0.313953
117,59,0.72093,0.489076,0.710944,0.965116,0.313953
91,46,0.72093,0.489076,0.710944,0.965116,0.313953
95,48,0.72093,0.489076,0.710944,0.965116,0.313953
113,57,0.72093,0.489076,0.710944,0.965116,0.313953
189,95,0.72093,0.489076,0.710944,0.965116,0.313953


### Leadtime LightGBM ??

Leadtime? ?? ????.

```text
?? holdout: accuracy 0.6512 / macro F1 0.4405 / top2 0.9651
?? holdout: accuracy 0.6744 / macro F1 0.4571 / top2 0.9651
```

???? ?? ??? ??? ??. ?? leadtime ??? ?? ?? ????? ??? ???? ?? pseudo label???, ? ??? ?????? ? ??. ?? ???? ? ? ???, ?? ?? ?? ? confusion matrix? bucket? ??? ? ? ? ???? ?? ??.


## 4. ?? ??

?? ??????? ??? ??? ??? ??.

- Risk LightGBM: ?? ?? ??. ?? ??? validation ???/holdout ??? ????.
- Leadtime LightGBM: ?? ??. ?? ??? ?? ????? ?? bucket? ?? ??.
- Isolation Forest: anomaly label ??? ?? ??. ?? ?? trade-off? ?? ?? ????? ??? ??.

?, ?? ????? ?? ?? ??? ?? Leadtime ?? ????, Risk? ? ?? ??? ??. Priority? ?? `v2_threshold48`? ?? ??? ??? ????.


## 5. ?? ?? ??? Leadtime bucket confusion

??? ???? ?? ??? ???????? ?? seed random search? ?? ????.

??? ?? ??? ??.

- validation?? ??? ??? holdout ?? ??? ??
- holdout??? ??? ?? ??? oracle ?????? ??
- ?? ??? validation ?? ??? ?? ???? holdout??? ??? ??


In [5]:
wide_iforest = read_csv("hyperparameter_tuning_wide_iforest_metrics.csv")
wide_risk = read_csv("hyperparameter_tuning_wide_risk_metrics.csv")
wide_lead = read_csv("hyperparameter_tuning_wide_leadtime_metrics.csv")
lead_conf = read_csv("leadtime_bucket_confusion_official_vs_tuned.csv")
wide_summary_path = REPORT_DIR / "hyperparameter_tuning_wide_summary.json"
wide_summary = json.loads(wide_summary_path.read_text(encoding="utf-8-sig")) if wide_summary_path.exists() else {}
wide_summary.keys()

dict_keys(['isolation_forest_wide', 'risk_lgbm_wide', 'leadtime_lgbm_wide'])

In [6]:
if not lead_conf.empty:
    display(lead_conf.rename(columns={
        "variant": "???",
        "actual_bucket": "?? bucket",
        "predicted_bucket": "?? bucket",
        "count": "??",
    }))
    fig = px.density_heatmap(
        lead_conf,
        x="predicted_bucket",
        y="actual_bucket",
        z="count",
        facet_col="variant",
        color_continuous_scale="Blues",
        title="Leadtime bucket confusion: ?? ?? vs ?? ?? validation ?? ??",
        labels={"predicted_bucket": "?? bucket", "actual_bucket": "?? bucket", "count": "??"},
    )
    fig.show()

if not wide_lead.empty:
    hold = wide_lead[wide_lead["split"].eq("holdout")].copy()
    top = hold.sort_values("macro_f1", ascending=False).head(10)
    display(top[["candidate_id", "accuracy", "macro_f1", "weighted_f1", "top2_accuracy", "bucket_distance_mae"]].rename(columns={
        "candidate_id": "?? ID", "accuracy": "???", "macro_f1": "Macro F1", "weighted_f1": "Weighted F1", "top2_accuracy": "Top2 ???", "bucket_distance_mae": "???? MAE"
    }))
    fig = px.scatter(
        wide_lead[wide_lead["split"].eq("validation")].merge(
            wide_lead[wide_lead["split"].eq("holdout")][["candidate_id", "macro_f1", "accuracy"]],
            on="candidate_id",
            suffixes=("_validation", "_holdout"),
        ),
        x="macro_f1_validation",
        y="macro_f1_holdout",
        hover_data=["candidate_id", "accuracy_validation", "accuracy_holdout"],
        title="?? Leadtime ??: Validation Macro F1 ?? Holdout Macro F1",
        labels={"macro_f1_validation": "Validation Macro F1", "macro_f1_holdout": "Holdout Macro F1"},
    )
    fig.show()

,???,?? bucket,?? bucket,??
0,official_promoted,0-24h,0-24h,25
1,official_promoted,0-24h,1-3d,10
2,official_promoted,0-24h,3-7d,0
3,official_promoted,1-3d,0-24h,17
4,official_promoted,1-3d,1-3d,31
5,official_promoted,1-3d,3-7d,0
6,official_promoted,3-7d,0-24h,3
7,official_promoted,3-7d,1-3d,0
8,official_promoted,3-7d,3-7d,0
9,wide_tuned_validation_selected,0-24h,0-24h,15


,?? ID,???,Macro F1,Weighted F1,Top2 ???,???? MAE
153,77,0.720930,0.484874,0.709746,0.965116,0.313953
589,295,0.709302,0.480994,0.699753,0.965116,0.325581
535,268,0.709302,0.480994,0.699753,0.965116,0.325581
251,126,0.709302,0.480453,0.699971,0.965116,0.325581
373,187,0.709302,0.478731,0.699513,0.965116,0.325581
189,95,0.709302,0.477541,0.698829,0.965116,0.325581
281,141,0.709302,0.476123,0.697834,0.965116,0.325581
279,140,0.697674,0.473522,0.687412,0.965116,0.337209
569,285,0.697674,0.473522,0.687412,0.965116,0.337209
497,249,0.697674,0.473522,0.687412,0.965116,0.337209


In [7]:
if not wide_risk.empty:
    merged = wide_risk[wide_risk["split"].eq("validation")][["candidate_id", "f1", "recall", "false_positive_rate"]].merge(
        wide_risk[wide_risk["split"].eq("holdout")][["candidate_id", "f1", "recall", "false_positive_rate", "roc_auc"]],
        on="candidate_id",
        suffixes=("_validation", "_holdout"),
    )
    display(merged.sort_values("f1_validation", ascending=False).head(10).rename(columns={
        "candidate_id": "?? ID", "f1_validation": "Validation F1", "f1_holdout": "Holdout F1", "recall_validation": "Validation Recall", "recall_holdout": "Holdout Recall", "false_positive_rate_validation": "Validation ???", "false_positive_rate_holdout": "Holdout ???", "roc_auc": "Holdout ROC-AUC"
    }))
    fig = px.scatter(
        merged,
        x="f1_validation",
        y="f1_holdout",
        hover_data=["candidate_id", "recall_validation", "recall_holdout", "false_positive_rate_validation", "false_positive_rate_holdout", "roc_auc"],
        title="?? Risk ??: Validation F1 ?? Holdout F1",
        labels={"f1_validation": "Validation F1", "f1_holdout": "Holdout F1"},
    )
    fig.show()

if not wide_iforest.empty:
    hold = wide_iforest[wide_iforest["split"].eq("holdout")].copy()
    fig = px.scatter(
        hold,
        x="recall",
        y="false_positive_rate",
        color="threshold_quantile",
        size="f1",
        hover_data=["candidate_id", "n_estimators", "max_samples", "max_features", "bootstrap", "precision"],
        title="?? Isolation Forest ??: Holdout ???? ??? trade-off",
        labels={"recall": "???", "false_positive_rate": "???", "threshold_quantile": "Threshold quantile", "f1": "F1"},
    )
    fig.show()

,?? ID,Validation F1,Validation Recall,Validation ???,Holdout F1,Holdout Recall,Holdout ???,Holdout ROC-AUC
54,55,0.762963,0.720280,0.083333,0.156028,0.127907,0.205607,0.527548
105,106,0.733096,0.720280,0.121528,0.351515,0.337209,0.233645,0.601880
149,150,0.716312,0.706294,0.131944,0.362573,0.360465,0.252336,0.605629
144,145,0.705882,0.629371,0.076389,0.222222,0.162791,0.121495,0.616333
115,116,0.704453,0.608392,0.059028,0.342105,0.302326,0.186916,0.589709
312,313,0.699647,0.692308,0.142361,0.231884,0.186047,0.168224,0.489024
226,227,0.696864,0.699301,0.152778,0.370370,0.348837,0.214953,0.626114
318,319,0.686667,0.720280,0.187500,0.400000,0.395349,0.233645,0.575581
142,143,0.686667,0.720280,0.187500,0.382022,0.395349,0.271028,0.564660
252,253,0.671186,0.692308,0.184028,0.348837,0.348837,0.261682,0.500706


### ?? ?? ?? ??

?? ??? ??? ? ?????.

```text
Risk wide validation ?? ??:
F1 0.1560 / Recall 0.1279 / FPR 0.2056 / ROC-AUC 0.5275
```

Risk? ? ?? ?? validation?? ?? ??? ??? holdout?? ????. ?? ????????? split/group drift? ?? ?? ??? ? ??? ???. Risk? ?? ?? ??? ??.

```text
Leadtime wide validation ?? ??:
Accuracy 0.6047 / Macro F1 0.3912 / Top2 0.9651
```

Leadtime? ?? ??? validation ?? ??? ???? ???. Holdout oracle?? Macro F1 0.4849 ??? ???, ?? holdout? ?? ?? ???? ?? ??? ? ? ??.

Bucket confusion? ?? ?? ??? 0-24h? 1-3d? ?? ?? ???? 3-7d? ?? ??? ???. ?? ?? validation ?? ??? 3-7d? ??? ???. ??? Leadtime ??? ?? ????????? 3-7d ?? ?, ?? ???, ?? 2?? urgency ??? ? ????.

Isolation Forest? F1? Recall? ? ????? FPR? ????. anomaly label ??? ???? ??? ??? ?? main ??? ??? ??.
